In [4]:
import json
import re
import os

notebook_path = r"C:\Users\Jotta-W\Documents\000 - MBA USP\POC_01.ipynb"
base_dir = r"C:\Users\Jotta-W\Documents\000 - MBA USP"

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

step_num = 1
for cell in nb.get('cells', []):
    if cell.get('cell_type') == 'code':
        source_lines = cell.get('source', [])
        if not source_lines:
            continue
            
        # Converte as linhas em texto e comenta comandos mágicos/shell do Jupyter (ex: !pip)
        processed_lines = []
        for line in source_lines:
            if line.strip().startswith(('!', '%')):
                processed_lines.append("# " + line)
            else:
                processed_lines.append(line)
        
        source_text = "".join(processed_lines)
        if not source_text.strip():
            continue
            
        # Determina o sufixo baseado no nome do notebook (idh ou pib)
        notebook_name = os.path.basename(notebook_path).lower()
        suffix = ""
        if "idh" in notebook_name:
            suffix = "-idh"
        elif "pib" in notebook_name:
            suffix = "-pib"
            
        filename = f"step-{step_num:02d}{suffix}.py"
        filepath = os.path.join(base_dir, filename)
        
        print(f"Escrevendo {filename}...")
        with open(filepath, 'w', encoding='utf-8') as out_f:
            out_f.write(source_text)
            
        step_num += 1

print("Concluído!")

Escrevendo step-01.py...
Escrevendo step-02.py...
Escrevendo step-03.py...
Escrevendo step-04.py...
Escrevendo step-05.py...
Escrevendo step-06.py...
Concluído!


In [1]:
# -------------------------------------------------------------
# Exporta cada celula de codigo de um notebook para um arquivo .py
# Nome: step-NN-sufixo.py
#   NN e sufixo sao lidos do comentario "# STEP NN SUFIXO" na celula
#   Ex: "# STEP 05 IDH - ..."  ->  step-05-idh.py
#   Fallback: contador sequencial se a celula nao tiver o comentario
# Comandos magicos/shell (!, %) sao comentados.
# Avisa (sem sobrescrever calado) se dois steps colidirem no mesmo nome.
# -------------------------------------------------------------
import json
import re
import os

notebook_path = r"C:\Users\Jotta-W\Documents\000 - MBA USP\POC_01.ipynb"
base_dir = r"C:\Users\Jotta-W\Documents\000 - MBA USP"

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

# captura "STEP 05 IDH" ou "STEP 05" (sufixo opcional)
padrao_step = re.compile(r'#\s*STEP\s+(\d+)\s*([A-Za-z]+)?', re.IGNORECASE)

usados = {}     # nome de arquivo -> de onde veio (para detectar colisao)
contador = 1

for cell in nb.get('cells', []):
    if cell.get('cell_type') != 'code':
        continue
    source_lines = cell.get('source', [])
    if not source_lines:
        continue

    # comenta comandos magicos/shell (ex: !pip, %matplotlib)
    processed_lines = [("# " + l) if l.strip().startswith(('!', '%')) else l
                       for l in source_lines]
    source_text = "".join(processed_lines)
    if not source_text.strip():
        continue

    achado = padrao_step.search(source_text)
    if achado:
        step_num = int(achado.group(1))
        suf = achado.group(2)
        suffix = f"-{suf.lower()}" if suf else ""
    else:
        step_num = contador
        suffix = ""

    filename = f"step-{step_num:02d}{suffix}.py"

    if filename in usados:
        print(f"AVISO: '{filename}' ja foi gerado. Pulando para nao sobrescrever.")
        contador += 1
        continue

    filepath = os.path.join(base_dir, filename)
    print(f"Escrevendo {filename}...")
    with open(filepath, 'w', encoding='utf-8') as out_f:
        out_f.write(source_text)
    usados[filename] = True
    contador += 1

print("Concluido!")

Escrevendo step-01.py...
Escrevendo step-02.py...
Escrevendo step-03.py...
Escrevendo step-04.py...
Escrevendo step-05-idh.py...
Escrevendo step-06-idh.py...
Escrevendo step-07-idh.py...
Escrevendo step-08-pib.py...
Escrevendo step-09-pib.py...
Escrevendo step-10-viz.py...
Escrevendo step-11.py...
Escrevendo step-12-viz.py...
Escrevendo step-13-viz.py...
Escrevendo step-14-viz.py...
Concluido!


In [5]:
# -------------------------------------------------------------
# STEP 08 PIB_AGRO - Matriz de correlacao das features do modelo
# Le a base com dummies do STEP 04, calcula a correlacao das 15
# features do modelo PIB_Agro (as 14 estruturais do IDH mais o
# proprio IDH como preditor), salva a matriz e sinaliza os pares
# com correlacao absoluta acima de 0,7 (criterio de multicolinearidade
# do TCC). Nao remove nada, apenas diagnostica.
# -------------------------------------------------------------
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('output/base_step_04.csv', sep=';', decimal=',', encoding='utf-8-sig')

# 15 features do modelo PIB_Agro
features_pibagro = [
    'IDH',
    'Pago_USD',
    'PIB',
    'Log_Pop_Por_Propriedade',
    '%_Receber_Orientacao_Tecnica',
    'tipoacao_CAPEX',
    'regiao_BR',
    'regiao_Centro-Oeste',
    'regiao_Norte',
    'regiao_Sudeste',
    'regiao_Sul',
    'unidadeorcamentaria_EMBRAPA',
    'unidadeorcamentaria_INCRA',
    'unidadeorcamentaria_MAPA',
    'unidadeorcamentaria_SFB'
]

for c in features_pibagro:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.'), errors='coerce')

corr = df[features_pibagro].corr()

corr.to_csv('output/base_step_08_corr_pibagro.csv', sep=';', encoding='utf-8-sig')

LIMIAR = 0.7
pares_altos = []
for i in range(len(features_pibagro)):
    for j in range(i + 1, len(features_pibagro)):
        valor = corr.iloc[i, j]
        if abs(valor) > LIMIAR:
            pares_altos.append((features_pibagro[i], features_pibagro[j], round(valor, 3)))

print(f"Matriz de correlacao PIB_Agro ({len(features_pibagro)} features)")
if pares_altos:
    print(f"Pares com |correlacao| > {LIMIAR} (potencial multicolinearidade):")
    for a, b, v in pares_altos:
        print(f"  {a} x {b}: {v}")
else:
    print(f"Nenhum par com |correlacao| > {LIMIAR}.")

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlacao - Features do Modelo PIB_Agro')
plt.tight_layout()
plt.savefig('output/base_step_08_corr_pibagro.png', dpi=150)
plt.close()

print("Salvo em output/base_step_08_corr_pibagro.csv e output/base_step_08_corr_pibagro.png")

Matriz de correlacao PIB_Agro (15 features)
Pares com |correlacao| > 0.7 (potencial multicolinearidade):
  IDH x %_Receber_Orientacao_Tecnica: 0.814
Salvo em output/base_step_08_corr_pibagro.csv e output/base_step_08_corr_pibagro.png
